# 🕳️ Clase 02 · Práctica guiada: ¿por qué faltan los datos?
Dataset: **viviendas en Lima** (sintético). Tiene faltantes de los tres tipos **puestos a propósito**.

El profesor explicó la teoría (MCAR, MAR, MNAR) con el ejemplo del IQ. Aquí haremos lo mismo con datos reales:
para cada columna con huecos preguntaremos **¿de qué depende que falte?**

Ejecuta celda por celda (`Shift + Enter`). En las celdas **✏️ Tu turno** escribes tú.
Al final implementa `ejercicio.py` y evalúa con `python -m framework evaluar <carpeta de esta clase>`.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repo, para que el notebook funcione desde cualquier carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "framework").exists())
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from framework.testing import ruta_dataset  # si el CSV no existe, lo genera

df = pd.read_csv(ruta_dataset("viviendas_lima.csv"))
df.shape

## Paso 0 · ¿Dónde faltan datos? (repaso de la clase 01)
Antes de preguntar *por qué*, hay que saber *dónde*. `df.isna().sum()` cuenta los NaN por columna.

In [ ]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
pd.DataFrame({"faltantes": faltantes, "porcentaje": (faltantes / len(df) * 100).round(2)})

Hay 4 columnas numéricas con huecos: `piso`, `antiguedad_anios`, `distancia_centro_km` y `banos`
(y dos categóricas, `estado` y `cochera`, que veremos en otra clase).

**La pregunta de hoy:** ¿estos huecos son al azar (MCAR) o dependen de algo (MAR / MNAR)?

## Paso 1 · Convertir "hay un hueco" en una columna
`df[col].isna()` devuelve una Serie de `True`/`False`. Si la guardamos como columna, podemos **agruparla y promediarla** como cualquier otra.

💡 **Truco clave:** la media de True/False es la **proporción de True** (True = 1, False = 0).

In [ ]:
ejemplo = pd.Series([True, False, False, False])
print("media:", ejemplo.mean(), "-> falta el", ejemplo.mean() * 100, "%")

In [ ]:
# Trabajamos sobre una COPIA para no ensuciar df
d = df.copy()
d["antiguedad_falta"] = d["antiguedad_anios"].isna()
d[["id", "tipo_inmueble", "antiguedad_anios", "antiguedad_falta"]].head(8)

## Paso 2 · MCAR: ¿la tasa de faltantes es parecida en todos los grupos?
Si `antiguedad_anios` falta **completamente al azar**, el % de huecos debería ser parecido en casas, departamentos y oficinas.

In [ ]:
tasa = d.groupby("tipo_inmueble")["antiguedad_falta"].mean() * 100
tasa.round(2)

Entre 7 % y 11 % en los tres tipos: diferencias pequeñas. **No hay un patrón**: compatible con **MCAR**.

Según el profesor, un MCAR con **pocos** faltantes (aquí ~8 %) se puede imputar con la mediana sin afectar mucho. Veamos cuánto cambia la distribución:

In [ ]:
mediana = df["antiguedad_anios"].median()
imputada = df["antiguedad_anios"].fillna(mediana)   # devuelve una Serie NUEVA, df no cambia

print("mediana usada:", mediana)
print("NaN antes:", df["antiguedad_anios"].isna().sum(), "| NaN después:", imputada.isna().sum())
pd.DataFrame({
    "original": df["antiguedad_anios"].describe(),
    "imputada": imputada.describe(),
}).round(2)

La media y los cuartiles casi no se mueven. La **std baja un poco**: imputar con un solo valor "aplana" la distribución.
Por eso el profesor dice que *"tampoco es tan bueno imputar con la mediana"*: funciona si son pocos.

## Paso 3 · MAR: la historia del IQ < 92, versión Lima
En el ejemplo del profesor faltaba el score **cuando el IQ era bajo**. ¿Hay alguna columna aquí cuyos huecos dependan de otra variable?

Miremos `distancia_centro_km` agrupando por `distrito`:

In [ ]:
d["distancia_falta"] = d["distancia_centro_km"].isna()
tasa_dist = (d.groupby("distrito")["distancia_falta"].mean() * 100).round(2).sort_values(ascending=False)
tasa_dist

In [ ]:
tasa_dist.plot(kind="barh", figsize=(7, 5), title="% de distancia_centro_km faltante por distrito")
plt.xlabel("% faltante")
plt.gca().invert_yaxis()
plt.show()

🔎 **Esto NO es al azar.** La distancia solo falta en 4 distritos (SJL, Ate, Los Olivos, Chorrillos), y en los demás nunca.
Es exactamente el patrón **MAR**: la ausencia depende de **otra variable observada** (`distrito`).

Como dijo el profesor: *"esto, a nivel de análisis exploratorio, **tienen que reportarlo**"*.
¿El culpable es el script o el registro? Quizás el sistema que calcula la distancia no tiene bien mapeados esos distritos.

**Rango** de la tasa (máximo − mínimo) = una forma rápida de medir *cuánto cambia según el grupo*:

In [ ]:
print("rango distancia por distrito:", round(tasa_dist.max() - tasa_dist.min(), 2), "puntos porcentuales")
print("rango antigüedad por tipo:   ", round(tasa.max() - tasa.min(), 2), "puntos porcentuales")

## ✏️ Tu turno 1 · El caso de `piso`
Crea la columna `piso_falta` en `d` y calcula el % de faltantes de `piso` por `tipo_inmueble`.
Luego responde en un comentario: ¿es MCAR o MAR? ¿Tiene **sentido** que falte? ¿Imputarías el piso de una casa con la mediana?

In [ ]:
# d["piso_falta"] = ...
# tasa_piso = ...
# tasa_piso


## Paso 4 · ¿Las filas con hueco son "distintas"? Comparar medias
Otra forma de buscar relación: separar las filas **con** y **sin** faltante y comparar otra variable (por ejemplo, el precio).

Una **máscara booleana** (True/False) dentro de `.loc[...]` filtra filas, y `~` la invierte.

In [ ]:
mascara = df["distancia_centro_km"].isna()          # True donde falta la distancia

precio_con_hueco = df.loc[mascara, "precio_usd"].mean()
precio_sin_hueco = df.loc[~mascara, "precio_usd"].mean()
print(f"precio medio CON distancia faltante: {precio_con_hueco:,.0f} USD")
print(f"precio medio SIN distancia faltante: {precio_sin_hueco:,.0f} USD")

Las viviendas sin distancia son **más baratas**, lo que cuadra con que sean de distritos periféricos.
📌 Para el modelo, que falte la distancia **trae información** sobre el precio.

## ✏️ Tu turno 2 · El misterio de `banos`
Haz la misma comparación para `banos`: precio medio de las filas **con** y **sin** dato de baños.

¿Qué observas? ¿Qué podría estar pasando cuando se registran los inmuebles caros?
¿Por qué sería peligroso imputar esos baños con la mediana?

In [ ]:
# mascara_banos = ...
# ...


## Paso 5 · MNAR: lo que no se ve
El profesor: MNAR es cuando *"hay datos faltantes, pero no entendemos el porqué"*. No hay relación visible con las otras variables, son **factores externos**.

Con las herramientas de hoy (tasa por grupo, comparación de medias) solo podemos ver relaciones con columnas **que tenemos**.
Si la causa es algo que no está en la tabla (el propio valor oculto, un proceso de registro…), estas herramientas no lo encuentran: hace falta **un estudio**, revisar la base de datos y hablar con quien registra.

📌 Extra: en `banos`, el generador del dataset simula que *los dueños de inmuebles caros ocultan el número de baños*. Que falte depende del **propio valor** (muchos baños = inmueble caro), y por eso se considera MNAR. Aquí lo vemos de forma indirecta gracias al precio. En la vida real a menudo no hay una columna que te avise.

## ✏️ Tu turno 3 · Tu reporte de faltantes
Completa la tabla (en esta celda markdown, haciendo doble clic) con lo que descubriste:

| Columna | % faltante | ¿Depende de…? | Tipo sospechado | ¿Imputar con mediana? |
|---|---|---|---|---|
| antiguedad_anios | | | | |
| distancia_centro_km | | | | |
| piso | | | | |
| banos | | | | |

Este reporte es justo lo que el profesor pide entregar en el EDA.

## Siguiente paso
Implementa las 5 funciones de `ejercicio.py`. Hacen de forma reutilizable lo que hiciste aquí a mano:
`agregar_indicador_faltante` → `imputar_mediana` → `tasa_faltantes_por_grupo` → `comparar_media_por_faltante` → `clasificar_mecanismo`.